# Import packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay

from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# uncomment when using Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Export directory

In [ ]:
export_dir = r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\TSS Modeling'

# Import data

In [ ]:
# import directory
data_directory = r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Landsat Sampling\Merged Landsat Data'

In [ ]:
# import data as dataframe
df = pd.read_csv(os.path.join(data_directory,'min_date.csv'))
df = df.drop(columns=['Unnamed: 0', 'CHLOROPHYLL',  
                      'CHLOROPHYLL_B', 'DOC', 'dif_date_point',
                      'N_TOTAL', 'N_TOTAL_DISSOLVED', 
                      'POC', 'P_ORGANIC', 'P_TOTAL', 
                      'SILICA',  'TOC', 'duplicated'],axis=1).rename(columns={'SPM':"TSS"})
df.columns

# Model Evaluation Functions

In [ ]:
def model_metrics(y_true,y_pred):
    ''' y = observed target values
    y_pred = predicted target values'''
    from sklearn.metrics import r2_score
    from sklearn.metrics import mean_absolute_error
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_percentage_error
    from sklearn.metrics import explained_variance_score

    return {'r2':r2_score(y_true, y_pred),
'mae':mean_absolute_error(y_true, y_pred),
'mse':mean_squared_error(y_true, y_pred),
'mape':mean_absolute_percentage_error(y_true, y_pred),
'exp_var': explained_variance_score(y_true, y_pred)
    }

In [ ]:
def cv_model_metrics(model,X,y,n_cv=20):
    ''' model = model to evaluate
    X = predictors
    y = observed target values
    cv = number of cross validations, standard is 5'''
    from sklearn.model_selection import ShuffleSplit

    cv = ShuffleSplit(n_splits=n_cv, test_size=0.15, random_state=0)
    from sklearn.model_selection import cross_val_score

    return {'r2':float(abs(cross_val_score(model, X, y, cv=cv,scoring='r2')).mean()),
'mae':float(abs(cross_val_score(model, X, y, cv=cv,scoring='neg_mean_absolute_error')).mean()),
'mse':float(abs(cross_val_score(model, X, y, cv=cv,scoring='neg_mean_squared_error')).mean()),
'mape':float(abs(cross_val_score(model, X, y, cv=cv,scoring='neg_mean_absolute_percentage_error')).mean()),
'exp_var':float( abs(cross_val_score(model, X, y, cv=cv,scoring='explained_variance')).mean())
    }

# Helper functions

**Helper Functions**
1. **`plot_evaluation()`** - Standardizes residual and prediction error plots
2. **`prepare_time_series_plot()`** - Creates consistent time-series visualizations with metrics
3. **`train_and_evaluate_model()`** - Unified training, evaluation, and plotting pipeline
4. **`batch_model_evaluation()`** - Batch processes multiple datasets with same model

In [ ]:
def plot_evaluation(y_true, y_pred, title="Model Evaluation"):
    """Create residual and prediction error plots."""
    fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
    PredictionErrorDisplay.from_predictions(
        y_true,
        y_pred=y_pred,
        kind="actual_vs_predicted",
        ax=axs[0]
    )
    axs[0].set_title("Actual vs. Predicted values")
    PredictionErrorDisplay.from_predictions(
        y_true,
        y_pred=y_pred,
        kind="residual_vs_predicted",
        ax=axs[1]
    )
    axs[1].set_title("Residuals vs. Predicted Values")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def prepare_time_series_plot(df, y_pred, cv_metrics_dict, title, r2_key='r2'):
    """Create time series plot with predictions and observations."""
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Plot predictions
    ax.plot(df['datetime'], y_pred, 
            color='gray', marker='x', linestyle='solid', linewidth=2, label='Predicted Model')
    
    # Define colors for water periods
    water_periods = sorted(df['WATER_PERIOD'].unique())
    color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    colors = {period: color_palette[i % len(color_palette)] for i, period in enumerate(water_periods)}
    
    # Plot observations by water period
    for water_period in water_periods:
        subset = df[df['WATER_PERIOD'] == water_period]
        ax.plot(subset['datetime'], subset['TSS'],
                marker='o', markersize=6, alpha=0.5, 
                color=colors[water_period], linestyle='none',
                label=f'{water_period}')
    # Set x-ticks
    total_len = len(y_pred)
    ticks = np.arange(0, total_len, 20)
    tick_labels = [df['date'].iloc[t] if t < len(df) else '' for t in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(tick_labels)
    
    # Add metrics text box
    # add sample size info 
    if cv_metrics_dict and r2_key in cv_metrics_dict:
        r2 = round(float(cv_metrics_dict[r2_key]), 4)
        r2_str = f'$R^2$ = {r2}'
        samples = f"Sample size: {total_len}"
        ax.text(0.05, 0.95, f"{r2_str}\n{samples}", transform=ax.transAxes, fontsize=10,
                verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

    


    ax.set_xlabel('Date')
    ax.set_ylabel('TSS (mg/L)')
    ax.set_title(title)
    ax.legend(loc='upper right')

   
    
    
    plt.tight_layout()
    plt.show()

In [ ]:
def train_and_evaluate_model(model, X, y, model_name, df_data=None, title=None):
    """Train model, evaluate, and optionally plot results."""
    # Fit and predict
    model_fit = model.fit(X, y)
    y_pred = model_fit.predict(X)
    
    # Calculate metrics
    metrics = model_metrics(y, y_pred)
    cv_metrics = cv_model_metrics(model, X, y, n_cv=20)
    
    print(f"\n{'='*50}")
    print(f"{model_name} Results")
    print(f"{'='*50}")
    print(f"Training Metrics: {metrics}")
    print(f"CV Metrics: {cv_metrics}")
    
    # Plot evaluation
    plot_evaluation(y, y_pred, title=f"{model_name} - Model Evaluation")
    
    # Plot time series if data provided
    if df_data is not None and title is not None:
        prepare_time_series_plot(df_data, y_pred, cv_metrics, title)
    
    return model_fit, y_pred, metrics, cv_metrics

# Model Configuration

In [ ]:
# Central model definitions
MODEL_CONFIG = {
    'ols': {
        'model': LinearRegression(),
        'description': 'Ordinary Least Squares'
    },
    'polynomial2': {
        'model': Pipeline([
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('linear', LinearRegression(positive=False))
        ]),
        'description': '2nd Degree Polynomial'
    },
    'polynomial3': {
        'model': Pipeline([
            ('poly', PolynomialFeatures(degree=3, include_bias=False)),
            ('linear', LinearRegression(positive=False))
        ]),
        'description': '3rd Degree Polynomial'
    },
    # 'splines': {
    #     'model': Pipeline([
    #         ('spline', SplineTransformer(n_knots=5, degree=3)),
    #         ('linear', LinearRegression(positive=False))
    #     ]),
    #     'description': 'Spline Transformation'
    # }
}

In [ ]:
def get_model(model_name):
    """
    Get model by name.
    Usage
    model_poly = get_model('polynomial')
    model_splines = get_model('splines')
    """
    return MODEL_CONFIG[model_name]['model']

In [ ]:
FEATURE_GROUPS = {
    'single_band': {
        'nir': {'band': ['nir_mean'],
                'name': 'NIR'},
        'red': {'band': ['red_mean'],
                'name': 'RED'},
        'green': {'band': ['green_mean'],
                  'name': 'GREEN'},
        'blue': {'band': ['blue_mean'],
                 'name': 'BLUE'},
        'nir_red_ratio': {'band': ['nir_red_ratio'],
                          'name': 'NIR/RED ratio'},
        'red_nir_ratio': {'band': ['red_nir_ratio'],
                          'name': 'RED/NIR ratio'}
    },
    'multi_band': {
        'nir_red': {'band': ['nir_mean', 'red_mean'],
                    'name': 'NIR and RED'},
        'all_bands': {'band': ['blue_mean', 'green_mean', 'red_mean', 'nir_mean'],
                      'name': 'All bands'}
    }
}

# Usage
# X_nir_red = df_subset[FEATURE_GROUPS['multi_band']['nir_red']]

# Cross-Validation Settings

In [ ]:
# CV Configuration
CV_CONFIG = {
    'n_splits': 20,
    'test_size': 0.20,
    'random_state': 666
}

# Use in helper function
# cv_metrics = cv_model_metrics(model, X, y, n_cv=CV_CONFIG['n_splits'])

# Visualization Configuration

In [ ]:
# Plot styling
PLOT_CONFIG = {
    'figsize': (16, 6),
    'colors': {
        'prediction': 'gray',
        'palette': ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
    },
    'marker_size': 6,
    'alpha': 0.5,
    'linewidth': 2,
    'font_size': 10
}

# TSS

## Data Preparation

In [ ]:
# filter parameters that will be used in the models
df_subset = df[['TSS','blue_mean',
       'green_mean',
       'nir_mean',
       'red_mean',
       'datetime',
       'WATER_PERIOD']].copy()
# remove empty values
df_subset = df_subset.dropna()
df_subset.isna().sum()

In [ ]:
# obtain date from datetime
df_subset['date'] = df_subset['datetime'].apply(lambda row: row[:10])
df_subset['nir_red_ratio'] = df_subset.apply(lambda row: float(row['nir_mean'] / row['red_mean']), axis=1)
df_subset['red_nir_ratio'] = df_subset.apply(lambda row: float(row['red_mean'] / row['nir_mean']), axis=1)

In [ ]:
y = df_subset['TSS'].copy()

# Fit model with all data

## Single Band

In [ ]:
for model_key in MODEL_CONFIG.keys():
    model = get_model(model_key)
    title = (MODEL_CONFIG[model_key]['description'])
    # Feature Exploration
    for features in FEATURE_GROUPS['single_band'].values():
        X = df_subset[features['band']]
        model_fit, y_pred, metrics, cv_metrics = train_and_evaluate_model(
            model=model,
            X=X, y=y,
            model_name=f"{title} - {features['name']}",
            title = f"{title} - {features['name']}",
            df_data=df_subset
        )

## Multiple bands

In [ ]:
for model_key in MODEL_CONFIG.keys():
    model = get_model(model_key)
    title = (MODEL_CONFIG[model_key]['description'])
    # Feature Exploration
    for features in FEATURE_GROUPS['multi_band'].values():
        X = df_subset[features['band']]
        model_fit, y_pred, metrics, cv_metrics = train_and_evaluate_model(
            model=model,
            X=X, y=y,
            model_name=f"{title} - {features['name']}",
            title = f"{title} - {features['name']}",
            df_data=df_subset
        )

# Fit model using stratification by Water Period

## Single band

In [ ]:
for water_period in df_subset['WATER_PERIOD'].unique():
    print(f"Water Period: {water_period}")
    df_filter = df_subset.loc[df_subset['WATER_PERIOD'] == water_period].copy()
    y_filter = df_filter['TSS'].copy()
    for model_key in MODEL_CONFIG.keys():
        model = get_model(model_key)
        title = (MODEL_CONFIG[model_key]['description'])
        # Feature Exploration
        for features in FEATURE_GROUPS['single_band'].values():
            X = df_filter[features['band']]
            model_fit, y_pred, metrics, cv_metrics = train_and_evaluate_model(
                model=model,
                X=X, y=y_filter,
                model_name=f"{title} - {features['name']}",
                title = f"{title} - {features['name']}",
                df_data=df_filter
            )

## Multiple bands

In [ ]:
for water_period in df_subset['WATER_PERIOD'].unique():
    print(f"Water Period: {water_period}")
    df_filter = df_subset.loc[df_subset['WATER_PERIOD'] == water_period].copy()
    y_filter = df_filter['TSS'].copy()
    for model_key in MODEL_CONFIG.keys():
        model = get_model(model_key)
        title = (MODEL_CONFIG[model_key]['description'])
        # Feature Exploration
        for features in FEATURE_GROUPS['multi_band'].values():
            X = df_filter[features['band']]
            model_fit, y_pred, metrics, cv_metrics = train_and_evaluate_model(
                model=model,
                X=X, y=y_filter,
                model_name=f"{title} - {features['name']}",
                title = f"{title} - {features['name']}",
                df_data=df_filter
            )